In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import pickle
from typing import Tuple, List, Optional, Dict
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, f1_score, classification_report
from sklearn.metrics import normalized_mutual_info_score, recall_score, precision_score
import warnings
warnings.filterwarnings('ignore')


# ==================== 1. 数据增强策略 ====================
class AdvancedSimCLRTransform:
    def __init__(self, image_size=96, strong_aug=True):
        self.image_size = image_size
        self.strong_aug = strong_aug
        self.mean = [0.4409, 0.4279, 0.3870]
        self.std = [0.2683, 0.2611, 0.2687]
        
    def __call__(self, x):
        if self.strong_aug:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.8,0.8,0.8,0.2)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.RandomApply([transforms.GaussianBlur(3)], p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.RandomApply([transforms.GaussianBlur(3)], p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
        else:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(self.image_size, scale=(0.2,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transform1
        return transform1(x), transform2(x)


# ==================== 2. 模型架构 ====================
class ImprovedResNetBackbone(nn.Module):
    def __init__(self, base_encoder='resnet50', pretrained=False):
        super().__init__()
        if base_encoder == 'resnet18':
            encoder = torchvision.models.resnet18
            self.global_dim = 512
            self.local_dim = 256
        elif base_encoder == 'resnet50':
            encoder = torchvision.models.resnet50
            self.global_dim = 2048
            self.local_dim = 1024
        else:
            raise ValueError(f"Unsupported base_encoder: {base_encoder}")
        
        try:
            self.backbone = encoder(weights="DEFAULT") if pretrained else encoder(weights=None)
        except:
            self.backbone = encoder(pretrained=pretrained)
        
        self.backbone.fc = nn.Identity()
        self.layer1 = self.backbone.layer1
        self.layer2 = self.backbone.layer2
        self.layer3 = self.backbone.layer3
        self.layer4 = self.backbone.layer4
    
    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        
        x_global = nn.functional.adaptive_avg_pool2d(x4, (1,1)).flatten(1)
        x_local = nn.functional.adaptive_avg_pool2d(x3, (4,4))
        return x_global, x_local


class ImprovedProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim=256, hidden_dim=2048):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return self.mlp(x)


class ImprovedSimCLRModel(nn.Module):
    # 全局 + 局部 双投影头
    def __init__(self, backbone, global_projector, local_projector):
        super().__init__()
        self.encoder = backbone
        self.global_projector = global_projector
        self.local_projector = local_projector
    
    def forward(self, x):
        global_feat, local_feat = self.encoder(x)
        
        # 全局投影
        z_global = self.global_projector(global_feat)
        z_global = nn.functional.normalize(z_global, dim=1)
        
        # 局部投影
        local_feat = local_feat.flatten(1)
        z_local = self.local_projector(local_feat)
        z_local = nn.functional.normalize(z_local, dim=1)
        
        return z_global, z_local, global_feat


# ==================== 3. 损失函数：联合损失 + 系数 ====================
def improved_simclr_loss(z1, z2, temperature=0.07):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / temperature
    
    labels = torch.cat([torch.arange(B) for _ in range(2)], dim=0).to(z.device)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    mask = torch.eye(2*B, device=z.device).bool()
    labels = labels.masked_fill(mask, 0)
    
    logits = sim - torch.max(sim, dim=1, keepdim=True)[0].detach()
    exp_logits = torch.exp(logits)
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
    loss = -(labels * log_prob).sum(1) / labels.sum(1)
    return loss.mean()


# 全局+局部联合损失，带权重系数 alpha, beta
def combined_feat_loss(z1_g, z2_g, z1_l, z2_l, temp=0.07, alpha=0.7, beta=0.3):
    loss_g = improved_simclr_loss(z1_g, z2_g, temp)
    loss_l = improved_simclr_loss(z1_l, z2_l, temp)
    total = alpha * loss_g + beta * loss_l
    return total, loss_g, loss_l


# ==================== 4. 训练 ====================
def train_epoch(model, loader, opt, device, epoch, alpha=0.7, beta=0.3, use_amp=True):
    model.train()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for v1, v2, _ in pbar:
        v1, v2 = v1.to(device), v2.to(device)
        opt.zero_grad()
        
        if use_amp:
            with torch.cuda.amp.autocast():
                z1_g, z1_l, _ = model(v1)
                z2_g, z2_l, _ = model(v2)
                loss, loss_g, loss_l = combined_feat_loss(z1_g,z2_g,z1_l,z2_l,0.07, alpha, beta)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            z1_g, z1_l, _ = model(v1)
            z2_g, z2_l, _ = model(v2)
            loss, loss_g, loss_l = combined_feat_loss(z1_g,z2_g,z1_l,z2_l,0.07, alpha, beta)
            loss.backward()
            opt.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss":f"{loss:.3f}", "g":f"{loss_g:.2f}", "l":f"{loss_l:.2f}"})
    return total_loss / len(loader)


# ==================== 5. 线性评估（修复完整版） ====================
def linear_evaluation_simple(model, train_loader, test_loader, num_classes=10, epochs=30, device='cuda'):
    model.eval()
    # 冻结自监督模型参数
    for p in model.parameters(): 
        p.requires_grad = False
    
    # 分类器
    linear = nn.Sequential(
        nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    ).to(device)
    
    opt = optim.AdamW(linear.parameters(), lr=0.01, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc = 0
    best_results = {}
    
    for ep in range(epochs):
        # 训练分类器
        linear.train()
        train_loss = 0.0
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                _, _, feat = model(x)
            logits = linear(feat)
            loss = criterion(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_loss += loss.item()
        
        # 测试评估
        linear.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                _, _, feat = model(x)
                preds = linear(feat).argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        
        # 计算指标
        acc = 100. * (np.array(all_preds) == np.array(all_labels)).mean()
        f1 = f1_score(all_labels, all_preds, average='macro')
        precision = precision_score(all_labels, all_preds, average='macro')
        recall = recall_score(all_labels, all_preds, average='macro')
        
        # 保存最优结果
        if acc > best_acc:
            best_acc = acc
            best_results = {
                'accuracy': acc,
                'f1': f1,
                'precision': precision,
                'recall': recall
            }
        
        print(f"Linear Ep{ep+1:2d} | Acc: {acc:.2f}% | Best: {best_acc:.2f}%")
        scheduler.step()
    
    # 打印最优指标
    print("\n" + "-"*50)
    print("Best Evaluation Results:")
    print(f"Accuracy:  {best_results['accuracy']:.2f}%")
    print(f"Macro F1:  {best_results['f1']:.4f}")
    print(f"Precision: {best_results['precision']:.4f}")
    print(f"Recall:    {best_results['recall']:.4f}")
    print("-"*50)
    
    return best_acc


# ==================== 主函数 ====================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # 超参数：调节全局/局部系数
    config = {
        'batch_size': 128,
        'lr': 3e-4,
        'epochs': 50,
        'alpha': 0.6,   # 全局损失权重
        'beta': 0.4,    # 局部损失权重
        'base_encoder':'resnet50',
        'pretrained':True,
        'use_amp':True,
    }
    
    os.makedirs('checkpoints_comb', exist_ok=True)
    
    # 数据
    image_size = 96
    class DualViewDataset:
        def __init__(self, ds, tfm): self.ds = ds; self.tfm = tfm
        def __len__(self): return len(self.ds)
        def __getitem__(self,i): img,l=self.ds[i]; v1,v2=self.tfm(img); return v1,v2,l
    
    train_ds = DualViewDataset(datasets.STL10('./data','unlabeled',download=True),
                               AdvancedSimCLRTransform(image_size,True))
    train_loader = DataLoader(train_ds, config['batch_size'], shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    
    # 模型：全局+局部双投影
    backbone = ImprovedResNetBackbone(config['base_encoder'], config['pretrained'])
    global_proj = ImprovedProjectionHead(backbone.global_dim, 256, 2048)
    local_proj = ImprovedProjectionHead(backbone.local_dim*4*4, 256, 2048)
    model = ImprovedSimCLRModel(backbone, global_proj, local_proj).to(device)
    
    # 优化
    opt = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=config['epochs'])
    
    # 训练
    losses = []
    for ep in range(config['epochs']):
        loss = train_epoch(model, train_loader, opt, device, ep+1,
                          config['alpha'], config['beta'], config['use_amp'])
        scheduler.step()
        losses.append(loss)
        print(f"Epoch {ep+1}/{config['epochs']} | Loss: {loss:.4f} | LR: {opt.param_groups[0]['lr']:.6f}")
    
    torch.save(model.state_dict(), 'checkpoints_comb/final_comb_model.pth')
    
    # 线性评估
    eval_tfm = transforms.Compose([transforms.Resize((image_size,image_size)), transforms.ToTensor(),
                                  transforms.Normalize([0.4409,0.4279,0.3870],[0.2683,0.2611,0.2687])])
    linear_train = DataLoader(datasets.STL10('./data','train',transform=eval_tfm), 128, shuffle=True, num_workers=4)
    linear_test = DataLoader(datasets.STL10('./data','test',transform=eval_tfm), 128, shuffle=False, num_workers=4)
    best_acc = linear_evaluation_simple(model, linear_train, linear_test, 10, 50, device)
    
    print("\n" + "="*50)
    print(f"FINAL BEST ACCURACY: {best_acc:.2f}%")
    print("="*50)


if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    main()

Using device: cuda
Files already downloaded and verified


Epoch 1: 100%|██████████| 781/781 [01:37<00:00,  8.01it/s, loss=2.099, g=2.07, l=2.14]


Epoch 1/50 | Loss: 2.4645 | LR: 0.000300


Epoch 2: 100%|██████████| 781/781 [01:34<00:00,  8.29it/s, loss=2.049, g=2.08, l=2.01]


Epoch 2/50 | Loss: 2.1439 | LR: 0.000299


...

Epoch 48/50 | Loss: 1.5441 | LR: 0.000001


Epoch 49: 100%|██████████| 781/781 [01:37<00:00,  7.99it/s, loss=1.563, g=1.56, l=1.57]


Epoch 49/50 | Loss: 1.5433 | LR: 0.000000


Epoch 50: 100%|██████████| 781/781 [01:34<00:00,  8.30it/s, loss=1.618, g=1.60, l=1.65]


Epoch 50/50 | Loss: 1.5373 | LR: 0.000000
Linear Ep 1 | Acc: 83.03% | Best: 83.03%
Linear Ep 2 | Acc: 84.19% | Best: 84.19%
Linear Ep 3 | Acc: 84.90% | Best: 84.90%
Linear Ep 4 | Acc: 85.29% | Best: 85.29%
Linear Ep 5 | Acc: 85.50% | Best: 85.50%
Linear Ep 6 | Acc: 85.67% | Best: 85.67%
Linear Ep 7 | Acc: 85.71% | Best: 85.71%
Linear Ep 8 | Acc: 85.89% | Best: 85.89%
Linear Ep 9 | Acc: 85.45% | Best: 85.89%
Linear Ep10 | Acc: 86.17% | Best: 86.17%
Linear Ep11 | Acc: 85.92% | Best: 86.17%
Linear Ep12 | Acc: 86.65% | Best: 86.65%
Linear Ep13 | Acc: 86.92% | Best: 86.92%
Linear Ep14 | Acc: 87.01% | Best: 87.01%
Linear Ep15 | Acc: 86.79% | Best: 87.01%
Linear Ep16 | Acc: 87.08% | Best: 87.08%
Linear Ep17 | Acc: 86.66% | Best: 87.08%
Linear Ep18 | Acc: 87.30% | Best: 87.30%
Linear Ep19 | Acc: 86.81% | Best: 87.30%
Linear Ep20 | Acc: 87.15% | Best: 87.30%
Linear Ep21 | Acc: 87.09% | Best: 87.30%
Linear Ep22 | Acc: 87.20% | Best: 87.30%
Linear Ep23 | Acc: 87.42% | Best: 87.42%
Linear Ep24 | A